#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when, sum, mean, count, countDistinct
from functools import reduce

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.prd_info")

In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos

def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()
    
    return

In [0]:
%skip
info(df)

#Transformations

## Rename columns' names

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"}
    
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Unique Values

In [0]:
%skip
info(df)

### product_id, product_number

In [0]:
%skip
#Explorando os valores duplicados

id = df.groupby("product_number").count()
id_filter = id.filter(col("count") > 1)
id_filter.display()

#Listando os codigos duplicados
lista_pn = [row.product_number for row in id_filter.select("product_number").collect()]
print(f"{len(lista_pn)} codigos duplucados")



In [0]:
%skip
#filtrando para verificar os casos de duplicidade

df_filter = df.filter(col("product_number").isin(lista_pn))

display(df_filter.limit(20))

df_filter.groupBy("product_number").agg(
    countDistinct("product_cost"),
    countDistinct("product_name")).display()

print("Os product_number estão duplicados porque têm custos diferentes. Então está ok.")

## Missings Values

In [0]:
%skip
info(df)

In [0]:
%skip
#Identificando as linhas em que há Nulo

df.withColumn(
    "Nulls",
    reduce(
        lambda a, b: a + b,
        [col(c).isNull().cast("int") for c in df.columns]
    )
).display()

In [0]:
df = df.fillna(0, subset=["product_cost"])
df = df.fillna("n/a", subset=["product_line"])


## Trimm

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
%skip
df.select(col("product_line")).distinct().display()

In [0]:
df = df.withColumn("product_line",
                   F.when(F.upper(col("product_line")) == "M", "Mountain")
                   .when(F.upper(col("product_line")) == "R", "Road")
                   .when(F.upper(col("product_line")) == "S", "Other Sales")
                   .when(F.upper(col("product_line")) == "T", "Touring")
                   .otherwise("n/a")
                   )

df.select(col("product_line")).distinct().display()

In [0]:
df = df.withColumn("end_date", col("end_date").cast("date"))



product_numb

In [0]:
df = df.withColumn("product_number",F.substring(col("product_number"),7,F.length(col("product_number"))))

#Write into Silver Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.crm_products")


In [0]:
%sql

SELECT * FROM workspace.silver.crm_products LIMIT 10